In [ ]:
# Check charge balance
from molmass import Formula
from re import split as resplit
from IORlib.utils import ppm2molL
from numpy import array, isfinite, negative
import matplotlib.pyplot as pl

from pathlib import Path
home = Path.home()
shared = Path('/home/shared/owner')
cop = shared/'CoP_files'
root = cop/'input/CF6_IORSIM_AH/CF6_IORSIM'

root = cop/'input/CF6_IORSIM_AH/CF6_IORSIM'
xls = ExcelFile(root.with_name('CF6_ions_v2.xlsx'))
conc = read_excel(xls, sheet_name='ions')
PV = conc['PV']

def charge(ion):
    specie = resplit('[+-]', ion)[0]
    return (specie, array(Formula(ion).charge * ppm2molL(specie) * conc[specie]))

def saveplot(name, echo=True):
    savename = root.parent/'plots'/name
    pl.savefig(savename) #, dpi=300)
    if echo:
        print(f'Saved {savename}')


ions = ('Na+', 'K+', 'Ca+2', 'Mg+2', 'Cl-', 'SO4-2')
charge = dict(charge(ion) for ion in ions)
total_charge = array([sum(val) for val in zip(*charge.values())])

pl.close('all')

# Plot charge 
pl.figure()
pl.title('Charge balance')
# Specie charge
for i, specie in enumerate(charge.keys()):
    chrg = charge[specie]
    mask = isfinite(chrg)
    line = '-'
    if chrg[0] < 0:
        chrg *= -1
        line = ':'
    pl.semilogy(PV[mask], chrg[mask], 'o', linestyle=line, mfc='w', mew=1.5, ms=4, alpha=0.5, label=ions[i])
# Total charge
mask = isfinite(total_charge)
tot_ch = total_charge[mask]
pv = PV[mask]
pl.semilogy(pv, abs(tot_ch), 'k-', label='Total charge')
positive = tot_ch > 0
negative = tot_ch < 0
pl.semilogy(pv[positive], tot_ch[positive], 'go', mec='k', alpha=0.7, label='Positive')
pl.semilogy(pv[negative], -tot_ch[negative], 'ro', mec='k', alpha=0.7, label='Negative')
pl.legend()
pl.xlabel('Pore volumes injected')
pl.ylabel('Charge per liter')
saveplot('plot_charge-balance.pdf')

# Plot concentration 
pl.figure()
pl.title('Measured concentrations, HCO3 inferred from charge balance')
pl.xlabel('Pore volumes injected')
pl.ylabel('mol/L')
# Specie charge
kwargs = {'mfc':'w', 'mew':1.5, 'ms':4, 'alpha':0.4}
for specie in charge.keys():
    mol_conc = ppm2molL(specie)*conc[specie]
    mask = isfinite(mol_conc)
    pl.semilogy(PV[mask], mol_conc[mask], '-o', **kwargs, label=specie)
# Infer HCO3 conc. from charge balance
mask = isfinite(total_charge)
tot_ch = total_charge[mask]
negative = tot_ch < 0
pv_hco3 = PV[mask][negative]
conc_hco3 = tot_ch[negative]/Formula('HCO3-').charge
kwargs['alpha'] = 1.0
kwargs['ms'] = 6
pl.semilogy(pv_hco3, conc_hco3, '-ko', **kwargs, label='HCO3')
pl.legend()
# pH
ax2 = pl.twinx()
mask = isfinite(conc['pH'])
ax2.plot(PV[mask], conc['pH'][mask], 'm:o', label='pH')
ax2.set_ylabel('pH')
pl.legend()
saveplot('plot_measured_conc_with_hco3.pdf')

# HCO3
pl.figure()
pl.title('HCO3 compared to simulations')
pl.semilogy(only_adv.species.PV, only_adv.species.conc['HCO3'], '-', mfc='None', label='Only advection')
pl.semilogy(felix_rates.species.PV, felix_rates.species.conc['HCO3'], '-', mfc='None', label='Std. rates')
pl.semilogy(pv_hco3, conc_hco3, 'o', mfc='None', mew=1.5, label='From charge balance')
pl.legend()
pl.xlabel('Pore volumes injected')
pl.ylabel('mol/L')
saveplot('plot_hco3_with_sim.pdf')

